# {{NOTEBOOK_TITLE}}

**Stage:** {{S1 | S2 | S3 | S4 | S5 | S6}}  
**Author:** {{subagent or human}}  
**Date:** {{YYYY-MM-DD}}  
**Run record:** `memory/runs/{{filename}}`

## Business context

{{One-paragraph reminder of the business ask. Read from `memory/business_context/`.}}

## What this notebook does

{{Bullet list — what reader should expect.}}

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

RUN_ID = datetime.utcnow().strftime('%Y-%m-%dT%H-%M')
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'CLAUDE.md').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MEMORY = PROJECT_ROOT / 'memory'
ARTIFACTS = PROJECT_ROOT / 'artifacts'
for sub in ('plots', 'models', 'reports'):
    (ARTIFACTS / sub).mkdir(parents=True, exist_ok=True)

print(f'Run ID:        {RUN_ID}')
print(f'Project root:  {PROJECT_ROOT}')
print(f'Python:        {sys.version.split()[0]}')
print(f'pandas:        {pd.__version__}')
print(f'numpy:         {np.__version__}')

## 2. Load data

In [ ]:
# Read data dictionary for this dataset before loading.
# Example:
# df = pd.read_csv(PROJECT_ROOT / 'data' / 'customers.csv')
# print(df.shape)
# df.head()
pass

## 3. Analysis

## 4. Sanity block

Standard theory-aware checks at the appropriate points in the analysis. See `docs/THEORY.md` for the math behind each.

Uncomment the checks that apply to this notebook's stage:
- **All stages with a design matrix `X`** → condition number (κ).
- **After PCA / dim reduction** → explained variance ratio.
- **After supervised fit** → bootstrap CI on the primary metric; check the metric is the one declared in `memory/business_context/`.
- **After clustering** → stability via ARI across seeds.
- **For any drift check** → invoke the `ds-distribution-comparison` family (KS / KL / MMD).

These checks correspond to invocations of the `ds-numerical-stability`, `ds-result-verification`, `ds-stat-sanity`, and `ds-distribution-comparison` skills.

In [ ]:
# --- Condition number check (uncomment when you have a design matrix X) ---
# kappa_X = np.linalg.cond(X)
# print(f'kappa(X)   = {kappa_X:.2e}')
# if kappa_X > 1e6:
#     print('⚠ kappa(X) > 1e6: matrix is ill-conditioned. Prefer SVD/QR solvers; consider ridge regularization.')
# elif kappa_X > 1e3:
#     print('Note: kappa(X) > 1e3 — use np.linalg.lstsq, not np.linalg.solve(X.T @ X, X.T @ y).')
# else:
#     print('kappa(X) is fine.')

# --- Explained variance after PCA ---
# from sklearn.decomposition import PCA
# pca = PCA().fit(X)
# cum = np.cumsum(pca.explained_variance_ratio_)
# k90 = int(np.searchsorted(cum, 0.90)) + 1
# k95 = int(np.searchsorted(cum, 0.95)) + 1
# print(f'k for 90% variance: {k90} / {X.shape[1]}')
# print(f'k for 95% variance: {k95} / {X.shape[1]}')

# --- Bootstrap CI on primary metric (binary classification example) ---
# from sklearn.utils import resample
# from sklearn.metrics import f1_score   # or whatever success_metric is
# scores = []
# for i in range(1000):
#     idx = resample(np.arange(len(y_test)), n_samples=len(y_test), random_state=i)
#     scores.append(f1_score(y_test[idx], y_pred[idx]))
# lo, hi = np.percentile(scores, [2.5, 97.5])
# print(f'F1 = {np.mean(scores):.3f} (95% CI [{lo:.3f}, {hi:.3f}])')

# --- Clustering stability via ARI across seeds ---
# from sklearn.cluster import KMeans
# from sklearn.metrics import adjusted_rand_score
# import itertools
# labels = [KMeans(n_clusters=k, random_state=s, n_init=10).fit(X).labels_ for s in range(10)]
# aris = [adjusted_rand_score(a, b) for a, b in itertools.combinations(labels, 2)]
# print(f'Stability (mean pairwise ARI): {np.mean(aris):.3f}  (low <0.7 means unstable)')

# --- Distribution comparison (drift / segment difference) ---
# from scipy.stats import ks_2samp
# stat, p = ks_2samp(sample_a, sample_b)
# print(f'KS stat = {stat:.4f}, p = {p:.4f}')

pass

## 5. Findings

{{Plain-language summary of what we learned. Will be copied into `memory/runs/`.}}

## 6. Run record

Write a structured record of this run to `memory/runs/`.

In [ ]:
run_record = {
    'run_id': RUN_ID,
    'stage': '{{S1 | S2 | S3 | S4 | S5 | S6}}',
    'notebook': '{{this notebook filename}}',
    'inputs': [],
    'outputs': [],
    'metrics': {},
    'random_seed': RANDOM_SEED,
    'env': {
        'python': sys.version.split()[0],
        'pandas': pd.__version__,
        'numpy': np.__version__,
    },
    'findings': '{{one-paragraph summary}}',
}

out = MEMORY / 'runs' / f'{RUN_ID}_{{run-slug}}.md'
out.write_text(
    '---\n' + json.dumps(run_record, indent=2) + '\n---\n\n'
    '# Run ' + RUN_ID + '\n\n'
    + run_record['findings'] + '\n'
)
print(f'Wrote {out}')